# Deepfake Detection — 

```
your_project_folder/
├── deepfake_detection_local.ipynb   ← الـ notebook ده
├── dataset_100/
│   ├── real/
│   ├── Deepfakes/
│   ├── Face2Face/
│   ├── FaceSwap/
│   └── NeuralTextures/
├── train.json
├── val.json
└── test.json
```

### Changes for Local Windows Execution
Change	Reason
BASE_DIR = os.getcwd()	Automatically sets paths based on the notebook location
num_workers = 0	Windows does not fully support multiprocessing in DataLoader
batch_size = 16	Reduces RAM usage when running on CPU
do_download = False	Dataset is already available locally
skip_extract = True	Frames are already extracted


# Deepfake Detection — Binary Baseline (Official FF++ Split)

This notebook uses the **official FaceForensics++ split files** (`train.json`, `val.json`, `test.json`) at the pair level.

### Key protocol choices
- **CPU-only** baseline
- **real = 0**, **fake = 1**
- **Official split** by pair membership (`<target>_<source>`)
- **Real videos** assigned using the original IDs appearing in each split file
- **WeightedRandomSampler** for balanced training
- **Threshold tuning** on validation
- **Frame-level** and **group-level** evaluation


In [1]:
import os, json, random, shutil, subprocess
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cpu')
print('DEVICE:', DEVICE)


DEVICE: cpu


In [6]:
import os

BASE_DIR = os.getcwd()

@dataclass
class CFG:
    # Paths — relative to notebook folder 
    root_dir: str = r"C:\Users\DPQUAI250101\Desktop\deepfake_baseline_model\FaceForensics_100\content\FaceForensics_100"
    work_dir:    str = BASE_DIR
    dataset_dir: str = os.path.join(BASE_DIR, 'dataset_100')
    index_csv:   str = os.path.join(BASE_DIR, 'final_officialsplit_index.csv')
    best_ckpt:   str = os.path.join(BASE_DIR, 'best_model_officialsplit.pth')

    # Split JSONs
    split_train_json: str = os.path.join(BASE_DIR, 'train.json')
    split_val_json:   str = os.path.join(BASE_DIR, 'val.json')
    split_test_json:  str = os.path.join(BASE_DIR, 'test.json')

    # Dataset
    compression:           str  = 'c23'
    max_videos_per_folder: int  = 100
    frames_per_video:      int  = 10
    image_size:            int  = 224

    # Download / Extract
    do_download:  bool = False   
    skip_extract: bool = True   

    download_script_url: str = 'https://kaldir.vc.in.tum.de/faceforensics_download_v4.py'
    server: str = 'EU2'

    # Training
    batch_size:    int   = 16    # قللناه لأن CPU
    num_workers:   int   = 0     # WINDOWS: لازم 0
    stage1_epochs: int   = 5
    stage2_epochs: int   = 15
    patience:      int   = 4
    stage1_lr:     float = 1e-3
    stage2_lr:     float = 1e-5  # FIX: was 5e-5
    weight_decay:  float = 1e-4
    label_smoothing: float = 0.05
    use_weighted_sampler: bool = True
    threshold_grid: int  = 39

cfg = CFG()
print("BASE_DIR:", BASE_DIR)
print("dataset_dir:", cfg.dataset_dir)
print("Config ready ✅")


BASE_DIR: c:\Users\DPQUAI250101\Desktop\deepfake_baseline_model
dataset_dir: c:\Users\DPQUAI250101\Desktop\deepfake_baseline_model\dataset_100
Config ready ✅


In [3]:
SPLIT_FILES = {
    'train': cfg.split_train_json,
    'val': cfg.split_val_json,
    'test': cfg.split_test_json,
}

def load_official_splits():
    split_pairs = {}
    for split_name, fp in SPLIT_FILES.items():
        data = json.loads(Path(fp).read_text())
        pair_keys = set()
        ids = set()
        for a, b in data:
            pair_keys.add(f'{a}_{b}')
            ids.add(a)
            ids.add(b)
        split_pairs[split_name] = {'pairs': pair_keys, 'ids': ids}
    return split_pairs

official_splits = load_official_splits()
print('Train pairs:', len(official_splits['train']['pairs']))
print('Val pairs:', len(official_splits['val']['pairs']))
print('Test pairs:', len(official_splits['test']['pairs']))
print('Sample train pairs:', list(sorted(official_splits['train']['pairs']))[:5])

# overlap sanity check
print('train∩val pairs:', len(official_splits['train']['pairs'] & official_splits['val']['pairs']))
print('train∩test pairs:', len(official_splits['train']['pairs'] & official_splits['test']['pairs']))
print('val∩test pairs:', len(official_splits['val']['pairs'] & official_splits['test']['pairs']))


Train pairs: 360
Val pairs: 70
Test pairs: 70
Sample train pairs: ['005_010', '006_002', '007_132', '011_805', '013_883']
train∩val pairs: 0
train∩test pairs: 0
val∩test pairs: 0


In [7]:
def run(cmd, check=True):
    print('>>', ' '.join(cmd))
    return subprocess.run(cmd, check=check)

def ensure_download_script():
    dst = Path(cfg.work_dir) / 'download.py'
    if dst.exists():
        return str(dst)
    run(['wget', '-O', str(dst), cfg.download_script_url, '--no-check-certificate'])
    return str(dst)

def download_videos():
    script = ensure_download_script()
    Path(cfg.root_dir).mkdir(parents=True, exist_ok=True)
    for d in ['original', 'Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']:
        run(['bash', '-lc', f'echo "" | python {script} {cfg.root_dir} -d {d} -c {cfg.compression} -t videos -n {cfg.max_videos_per_folder} --server {cfg.server}'])

def list_mp4(folder):
    return sorted([p for p in Path(folder).glob('*.mp4')])

def sanity_check_videos():
    root = Path(cfg.root_dir)
    paths = {
        'real': root / f'original_sequences/youtube/{cfg.compression}/videos',
        'Deepfakes': root / f'manipulated_sequences/Deepfakes/{cfg.compression}/videos',
        'Face2Face': root / f'manipulated_sequences/Face2Face/{cfg.compression}/videos',
        'FaceSwap': root / f'manipulated_sequences/FaceSwap/{cfg.compression}/videos',
        'NeuralTextures': root / f'manipulated_sequences/NeuralTextures/{cfg.compression}/videos',
    }
    for k,p in paths.items():
        vids = list_mp4(str(p))
        print(f'{k}: {len(vids)} videos | sample: {[v.name for v in vids[:3]]}')

if cfg.do_download:
    download_videos()

sanity_check_videos()


real: 100 videos | sample: ['033.mp4', '035.mp4', '036.mp4']
Deepfakes: 100 videos | sample: ['033_097.mp4', '035_036.mp4', '036_035.mp4']
Face2Face: 100 videos | sample: ['033_097.mp4', '035_036.mp4', '036_035.mp4']
FaceSwap: 100 videos | sample: ['033_097.mp4', '035_036.mp4', '036_035.mp4']
NeuralTextures: 100 videos | sample: ['033_097.mp4', '035_036.mp4', '036_035.mp4']


In [5]:
def build_face_cascade():
    haar = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    return cv2.CascadeClassifier(haar)

def center_crop_square(img):
    h, w = img.shape[:2]
    m = min(h, w)
    y1 = (h-m)//2
    x1 = (w-m)//2
    return img[y1:y1+m, x1:x1+m]

def detect_face(frame, cascade, pad=0.40):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = cascade.detectMultiScale(gray, 1.1, 5, minSize=(60,60))
    if len(faces) == 0:
        return None
    x, y, w, h = sorted(faces, key=lambda b: b[2]*b[3], reverse=True)[0]
    cx, cy = x + w/2, y + h/2
    side = max(w, h) * (1 + pad)
    x1 = int(max(0, cx - side/2)); y1 = int(max(0, cy - side/2))
    x2 = int(min(frame.shape[1], cx + side/2)); y2 = int(min(frame.shape[0], cy + side/2))
    crop = frame[y1:y2, x1:x2]
    return crop if crop.size > 0 else None

def extract_frames(video_dir, out_dir):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    videos = sorted(Path(video_dir).glob('*.mp4'))[:cfg.max_videos_per_folder]
    cascade = build_face_cascade()
    total = 0
    for vp in videos:
        cap = cv2.VideoCapture(str(vp))
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if n <= 0:
            cap.release(); continue
        step = max(1, n // cfg.frames_per_video)
        base = vp.stem
        for i in range(cfg.frames_per_video):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
            ok, frame = cap.read()
            if not ok:
                continue
            crop = detect_face(frame, cascade)
            if crop is None:
                crop = center_crop_square(frame)
            crop = cv2.resize(crop, (cfg.image_size, cfg.image_size))
            out_path = Path(out_dir) / f'{base}_frame{i:02d}.jpg'
            cv2.imwrite(str(out_path), crop)
            total += 1
        cap.release()
    print(f'{out_dir} -> {total} frames')

def extract_all_frames():
    out_root = Path(cfg.dataset_dir)
    out_root.mkdir(parents=True, exist_ok=True)
    if cfg.skip_extract:
        existing = sum(1 for _ in out_root.rglob('*.jpg'))
        if existing > 0:
            print(f'Frames already exist: {existing} jpg files. Skipping extraction.')
            return
    root = Path(cfg.root_dir)
    extract_frames(str(root / f'original_sequences/youtube/{cfg.compression}/videos'), str(out_root / 'real'))
    for m in ['Deepfakes','Face2Face','FaceSwap','NeuralTextures']:
        extract_frames(str(root / f'manipulated_sequences/{m}/{cfg.compression}/videos'), str(out_root / m))

extract_all_frames()


Frames already exist: 5000 jpg files. Skipping extraction.


In [7]:
def video_id_from_name(fname):
    return fname.split('_frame')[0]

def build_index_official():
    data_root = Path(cfg.dataset_dir)
    rows = []
    classes = sorted([d.name for d in data_root.iterdir() if d.is_dir()])
    for cls in classes:
        cls_dir = data_root / cls
        for p in cls_dir.glob('*.jpg'):
            vid = video_id_from_name(p.name)
            bin_label = 'real' if cls == 'real' else 'fake'
            target = 0 if bin_label == 'real' else 1
            assigned_split = None
            if bin_label == 'fake':
                if vid in official_splits['train']['pairs']:
                    assigned_split = 'train'
                elif vid in official_splits['val']['pairs']:
                    assigned_split = 'val'
                elif vid in official_splits['test']['pairs']:
                    assigned_split = 'test'
            else:
                if vid in official_splits['train']['ids']:
                    assigned_split = 'train'
                elif vid in official_splits['val']['ids']:
                    assigned_split = 'val'
                elif vid in official_splits['test']['ids']:
                    assigned_split = 'test'
            if assigned_split is None:
                continue
            rows.append({
                'path': str(p),
                'label': cls,
                'video_id': vid,
                'bin_label': bin_label,
                'target': target,
                'split_group': vid,
                'video_uid': f'{cls}::{vid}',
                'split': assigned_split,
            })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError('No frames matched the official split. Check split files and extraction.')
    df.to_csv(cfg.index_csv, index=False)
    print('Saved index:', cfg.index_csv)
    print('Frame counts by split/class:')
    print(df.groupby(['split','bin_label']).size())
    print('Unique groups by split/class:')
    print(df.groupby(['split','bin_label'])['split_group'].nunique())
    return df

df = build_index_official()

# Sanity checks
train_groups = set(df[df.split == 'train']['split_group'].unique())
val_groups = set(df[df.split == 'val']['split_group'].unique())
test_groups = set(df[df.split == 'test']['split_group'].unique())
print('train ∩ val :', len(train_groups & val_groups))
print('train ∩ test:', len(train_groups & test_groups))
print('val ∩ test  :', len(val_groups & test_groups))


Saved index: c:\Users\DPQUAI250101\Desktop\deepfake_baseline_model\final_officialsplit_index.csv
Frame counts by split/class:
split  bin_label
test   fake          360
       real          180
train  fake         1240
       real          620
val    fake          400
       real          200
dtype: int64
Unique groups by split/class:
split  bin_label
test   fake          9
       real         18
train  fake         31
       real         62
val    fake         10
       real         20
Name: split_group, dtype: int64
train ∩ val : 0
train ∩ test: 0
val ∩ test  : 0


In [8]:
class FFPPDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        label = int(row['target'])
        if self.transform:
            img = self.transform(img)
        return img, label

def make_loaders(df):
    train_tf = transforms.Compose([
        transforms.Resize((cfg.image_size, cfg.image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    test_tf = transforms.Compose([
        transforms.Resize((cfg.image_size, cfg.image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    train_df = df[df.split == 'train'].copy()
    val_df = df[df.split == 'val'].copy()
    test_df = df[df.split == 'test'].copy()
    train_ds = FFPPDataset(train_df, train_tf)
    val_ds = FFPPDataset(val_df, test_tf)
    test_ds = FFPPDataset(test_df, test_tf)
    if cfg.use_weighted_sampler:
        labels = train_df['target'].values
        counts = np.bincount(labels, minlength=2)
        class_weights = 1.0 / np.maximum(counts, 1)
        sample_weights = class_weights[labels]
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler, num_workers=cfg.num_workers)
    else:
        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
    print('Val distribution:')
    print(val_df['bin_label'].value_counts())
    return train_loader, val_loader, test_loader, val_df, test_df, test_tf

train_loader, val_loader, test_loader, val_df, test_df, test_tf = make_loaders(df)


Train: 1860 | Val: 600 | Test: 540
Val distribution:
bin_label
fake    400
real    200
Name: count, dtype: int64


In [9]:
def build_model():
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    for p in model.features.parameters():
        p.requires_grad = False
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(model.last_channel, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 2),
    )
    model.to(DEVICE)
    return model

def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total_loss = 0.0
    preds, trues, prob_fake_list = [], [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train:
                optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * y.size(0)
            pred = out.argmax(1)
            preds += pred.detach().cpu().numpy().tolist()
            trues += y.detach().cpu().numpy().tolist()
            prob_fake = torch.softmax(out, dim=1)[:, 1]
            prob_fake_list += prob_fake.detach().cpu().numpy().tolist()
    acc = accuracy_score(trues, preds)
    mf1 = f1_score(trues, preds, average='macro')
    auc = roc_auc_score(trues, prob_fake_list) if len(set(trues)) == 2 else float('nan')
    return total_loss / len(loader.dataset), acc, mf1, auc, trues, preds, prob_fake_list

def tune_threshold(y_true, prob_fake, grid=39, metric='accuracy'):
    best_t, best_score, best_acc, best_f1 = 0.5, -1, -1, -1
    for t in np.linspace(0.05, 0.95, grid):
        pred = (np.array(prob_fake) >= t).astype(int)
        acc = accuracy_score(y_true, pred)
        mf1 = f1_score(y_true, pred, average='macro')
        score = acc if metric == 'accuracy' else mf1
        if score > best_score:
            best_t, best_score, best_acc, best_f1 = float(t), score, acc, mf1
    return best_t, best_acc, best_f1

def collect_group_scores(model, df_split, transform):
    model.eval()
    group_scores, group_true = {}, {}
    with torch.no_grad():
        for _, row in df_split.iterrows():
            img = Image.open(row['path']).convert('RGB')
            x = transform(img).unsqueeze(0).to(DEVICE)
            out = model(x)
            prob_fake = torch.softmax(out, dim=1)[0, 1].cpu().item()
            g = row['split_group']
            group_scores.setdefault(g, []).append(prob_fake)
            group_true[g] = int(row['target'])
    y_true, y_score = [], []
    for g, scores in group_scores.items():
        y_true.append(group_true[g])
        y_score.append(float(np.mean(scores)))
    return np.array(y_true), np.array(y_score)

def eval_group_level(model, df_split, transform, threshold):
    y_true, y_score = collect_group_scores(model, df_split, transform)
    y_pred = (y_score >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, average='macro')
    auc = roc_auc_score(y_true, y_score) if len(set(y_true)) == 2 else float('nan')
    return acc, mf1, auc, len(y_true)

model = build_model()

class_w   = torch.tensor([4.0, 1.0], dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=cfg.label_smoothing)
print('Model ready. Trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

optimizer = optim.AdamW(model.classifier.parameters(), lr=cfg.stage1_lr, weight_decay=cfg.weight_decay)
print('=== Stage 1 ===')
for ep in range(cfg.stage1_epochs):
    tr = run_epoch(model, train_loader, criterion, optimizer)
    va = run_epoch(model, val_loader, criterion)
    print(f'[S1] Ep{ep+1}/{cfg.stage1_epochs} | Train acc {tr[1]*100:.1f}% f1 {tr[2]:.3f} auc {tr[3]:.3f} | Val acc {va[1]*100:.1f}% f1 {va[2]:.3f} auc {va[3]:.3f}')


for p in model.features[-4:].parameters():  
    p.requires_grad = True
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg.stage2_lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
best_f1, bad = -1.0, 0
print('=== Stage 2 ===')
for ep in range(cfg.stage2_epochs):
    tr = run_epoch(model, train_loader, criterion, optimizer)
    va = run_epoch(model, val_loader, criterion)
    scheduler.step(va[2])
    if va[2] > best_f1:
        best_f1 = va[2]
        bad = 0
        torch.save(model.state_dict(), cfg.best_ckpt)
    else:
        bad += 1
    print(f'[S2] Ep{ep+1:02d} | Train acc {tr[1]*100:.1f}% f1 {tr[2]:.3f} | Val acc {va[1]*100:.1f}% f1 {va[2]:.3f} auc {va[3]:.3f} | best {best_f1:.3f} bad {bad}/{cfg.patience}')
    if bad >= cfg.patience:
        print('Early stopping')
        break

model.load_state_dict(torch.load(cfg.best_ckpt, map_location=DEVICE))
va = run_epoch(model, val_loader, criterion)
y_val = np.array(va[4]); p_val = np.array(va[6])
best_t_acc, val_acc_tuned, val_f1_at_acc = tune_threshold(y_val, p_val, grid=cfg.threshold_grid, metric='accuracy')
best_t_f1, val_acc_at_f1, val_f1_tuned = tune_threshold(y_val, p_val, grid=cfg.threshold_grid, metric='f1')
print('Best threshold for Val Accuracy:', best_t_acc)
print('Val Accuracy:', val_acc_tuned)
print('Val Macro-F1:', val_f1_at_acc)
print('Best threshold for Val Macro-F1:', best_t_f1)
print('Val Accuracy:', val_acc_at_f1)
print('Val Macro-F1:', val_f1_tuned)

threshold = best_t_acc
te = run_epoch(model, test_loader, criterion)
y_test = np.array(te[4]); p_test = np.array(te[6])
pred_test = (p_test >= threshold).astype(int)
test_acc = accuracy_score(y_test, pred_test)
test_f1 = f1_score(y_test, pred_test, average='macro')
test_auc = roc_auc_score(y_test, p_test) if len(set(y_test)) == 2 else float('nan')
print('=== TEST RESULTS ===')
print(f'Accuracy: {test_acc*100:.2f}%')
print(f'Macro-F1: {test_f1:.3f}')
print(f'AUC: {test_auc:.3f}')

print('Confusion Matrix:\n', confusion_matrix(y_test, pred_test))
print(classification_report(y_test, pred_test, target_names=['real','fake']))

val_group = eval_group_level(model, val_df, test_tf, threshold)
test_group = eval_group_level(model, test_df, test_tf, threshold)
print('=== GROUP-LEVEL RESULTS ===')
print(f'VAL  acc {val_group[0]*100:.2f}% | f1 {val_group[1]:.3f} | auc {val_group[2]:.3f} | groups {val_group[3]}')
print(f'TEST acc {test_group[0]*100:.2f}% | f1 {test_group[1]:.3f} | auc {test_group[2]:.3f} | groups {test_group[3]}')

Model ready. Trainable params: 328450
=== Stage 1 ===
[S1] Ep1/5 | Train acc 50.0% f1 0.343 auc 0.654 | Val acc 36.5% f1 0.312 auc 0.679
[S1] Ep2/5 | Train acc 54.3% f1 0.426 auc 0.745 | Val acc 33.8% f1 0.262 auc 0.684
[S1] Ep3/5 | Train acc 60.3% f1 0.538 auc 0.797 | Val acc 55.7% f1 0.557 auc 0.728
[S1] Ep4/5 | Train acc 60.0% f1 0.536 auc 0.795 | Val acc 43.0% f1 0.409 auc 0.730
[S1] Ep5/5 | Train acc 64.6% f1 0.603 auc 0.837 | Val acc 56.5% f1 0.565 auc 0.719
=== Stage 2 ===
[S2] Ep01 | Train acc 71.9% f1 0.702 | Val acc 52.5% f1 0.523 auc 0.744 | best 0.523 bad 0/4
[S2] Ep02 | Train acc 67.1% f1 0.650 | Val acc 51.5% f1 0.509 auc 0.760 | best 0.523 bad 1/4
[S2] Ep03 | Train acc 69.8% f1 0.677 | Val acc 51.2% f1 0.506 auc 0.769 | best 0.523 bad 2/4
[S2] Ep04 | Train acc 68.6% f1 0.662 | Val acc 47.2% f1 0.459 auc 0.760 | best 0.523 bad 3/4
[S2] Ep05 | Train acc 69.6% f1 0.670 | Val acc 49.8% f1 0.489 auc 0.774 | best 0.523 bad 4/4
Early stopping
Best threshold for Val Accuracy: 0.

In [10]:
import io
import numpy as np
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import torch
 

 
def apply_jpeg_compression(img: Image.Image, quality: int) -> Image.Image:
    
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).convert('RGB')
 
def apply_gaussian_noise(img: Image.Image, std: float) -> Image.Image:
    
    arr = np.array(img, dtype=np.float32)
    noise = np.random.randn(*arr.shape) * std
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)
 

 
def eval_robustness(model, df_split, transform, threshold, perturb_fn):
    
    model.eval()
    preds, trues, probs = [], [], []
    with torch.no_grad():
        for _, row in df_split.iterrows():
            img = Image.open(row['path']).convert('RGB')
            img = perturb_fn(img)                        # ← التشويش هنا
            x = transform(img).unsqueeze(0).to(DEVICE)
            out = model(x)
            prob_fake = torch.softmax(out, dim=1)[0, 1].cpu().item()
            pred = int(prob_fake >= threshold)
            probs.append(prob_fake)
            preds.append(pred)
            trues.append(int(row['target']))
 
    acc  = accuracy_score(trues, preds)
    mf1  = f1_score(trues, preds, average='macro')
    auc  = roc_auc_score(trues, probs) if len(set(trues)) == 2 else float('nan')
    return acc, mf1, auc
 

 
jpeg_qualities = [80, 60, 40, 20]          
gaussian_stds  = [5, 10, 20, 40]           
 
results = []
 
# Baseline 
acc, mf1, auc = eval_robustness(
    model, test_df, test_tf, threshold,
    perturb_fn=lambda img: img
)
results.append({'perturbation': 'None (baseline)', 'param': '-', 'acc': acc, 'f1': mf1, 'auc': auc})
print(f"{'Perturbation':<30} {'Param':<8} {'Acc':>7} {'F1':>7} {'AUC':>7}")
print('-' * 65)
print(f"{'None (baseline)':<30} {'-':<8} {acc*100:>6.2f}% {mf1:>7.3f} {auc:>7.3f}")
 
# JPEG Compression
for q in jpeg_qualities:
    acc, mf1, auc = eval_robustness(
        model, test_df, test_tf, threshold,
        perturb_fn=lambda img, q=q: apply_jpeg_compression(img, q)
    )
    results.append({'perturbation': 'JPEG Compression', 'param': f'q={q}', 'acc': acc, 'f1': mf1, 'auc': auc})
    print(f"{'JPEG Compression':<30} {f'q={q}':<8} {acc*100:>6.2f}% {mf1:>7.3f} {auc:>7.3f}")
 
# Gaussian Noise
for s in gaussian_stds:
    acc, mf1, auc = eval_robustness(
        model, test_df, test_tf, threshold,
        perturb_fn=lambda img, s=s: apply_gaussian_noise(img, s)
    )
    results.append({'perturbation': 'Gaussian Noise', 'param': f'std={s}', 'acc': acc, 'f1': mf1, 'auc': auc})
    print(f"{'Gaussian Noise':<30} {f'std={s}':<8} {acc*100:>6.2f}% {mf1:>7.3f} {auc:>7.3f}")
 

 
import pandas as pd
rob_df = pd.DataFrame(results)
rob_csv = os.path.join(BASE_DIR, 'robustness_results.csv')
rob_df.to_csv(rob_csv, index=False)
print(f'\the result is saved: {rob_csv}')

Perturbation                   Param        Acc      F1     AUC
-----------------------------------------------------------------
None (baseline)                -         64.63%   0.604   0.637
JPEG Compression               q=80      60.37%   0.589   0.642
JPEG Compression               q=60      51.85%   0.515   0.583
JPEG Compression               q=40      49.81%   0.498   0.569
JPEG Compression               q=20      55.93%   0.553   0.596
Gaussian Noise                 std=5     67.96%   0.441   0.612
Gaussian Noise                 std=10    66.67%   0.400   0.503
Gaussian Noise                 std=20    66.67%   0.400   0.422
Gaussian Noise                 std=40    66.67%   0.400   0.440
	he result is saved: c:\Users\DPQUAI250101\Desktop\deepfake_baseline_model\robustness_results.csv


In [11]:
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms

# ─────────────────────────────────────────────
# 1. Save the model
# ─────────────────────────────────────────────

save_path = os.path.join(BASE_DIR, 'deepfake_model_final.pth')
torch.save(model.state_dict(), save_path)
print(f'Model saved to: {save_path}')


# ─────────────────────────────────────────────
# 2. Load the model from disk
# ─────────────────────────────────────────────

def load_model(path):
    m = build_model()
    m.load_state_dict(torch.load(path, map_location=DEVICE))
    m.eval()
    print(f'Model loaded from: {path}')
    return m

# To reload the model at any time:
# model = load_model(save_path)


# ─────────────────────────────────────────────
# 3. Run inference on a single image
# ─────────────────────────────────────────────

def predict_image(img_path, model, threshold=0.5):
    """
    Load an image from disk and predict whether it is real or fake.
    img_path  : path to the image file (jpg/png)
    model     : loaded model
    threshold : decision threshold (default 0.5, or use best_t_acc)
    """
    tf = transforms.Compose([
        transforms.Resize((cfg.image_size, cfg.image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])
    img = Image.open(img_path).convert('RGB')
    x   = tf(img).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        out       = model(x)
        probs     = torch.softmax(out, dim=1)[0]
        prob_real = probs[0].item()
        prob_fake = probs[1].item()
        label     = 'FAKE' if prob_fake >= threshold else 'REAL'

    print(f'Image     : {os.path.basename(img_path)}')
    print(f'Prediction: {label}')
    print(f'prob_real = {prob_real:.4f}  |  prob_fake = {prob_fake:.4f}')
    print(f'threshold = {threshold:.3f}')
    return label, prob_real, prob_fake


# ─────────────────────────────────────────────
# 4. Run on an image — change the path below
# ─────────────────────────────────────────────

img_path = r"C:\Users\DPQUAI250101\Downloads\image (3).png"  

if os.path.exists(img_path):
    predict_image(img_path, model, threshold=threshold)
else:
    print(f'Image not found: {img_path}')
    print('Update img_path to a valid image on your machine.')

Model saved to: c:\Users\DPQUAI250101\Desktop\deepfake_baseline_model\deepfake_model_final.pth
Image     : image (3).png
Prediction: FAKE
prob_real = 0.6398  |  prob_fake = 0.3602
threshold = 0.168


In [12]:
predict_image(img_path, model, threshold=threshold)


predict_image(img_path, model, threshold=0.5)          # neutral
predict_image(img_path, model, threshold=best_t_f1)    # optimized for F1

Image     : image (3).png
Prediction: FAKE
prob_real = 0.6398  |  prob_fake = 0.3602
threshold = 0.168
Image     : image (3).png
Prediction: REAL
prob_real = 0.6398  |  prob_fake = 0.3602
threshold = 0.500
Image     : image (3).png
Prediction: FAKE
prob_real = 0.6398  |  prob_fake = 0.3602
threshold = 0.192


('FAKE', 0.6398187279701233, 0.3601812422275543)

In [17]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── colour palette matching IEEE paper style ──────────────────────────────────
BLUE    = "#1f77b4"
ORANGE  = "#ff7f0e"
GREEN   = "#2ca02c"
RED     = "#d62728"
GREY    = "#7f7f7f"
STAGE_DIV_COLOR = "#aaaaaa"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIGURE 1 — Confusion Matrix
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
cm = np.array([[86, 94],
               [97, 263]])

labels = ["Real", "Fake"]
total  = cm.sum()
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100   # row-normalised %

fig1, ax = plt.subplots(figsize=(5.2, 4.4))

# background heatmap (row-normalised)
sns.heatmap(
    cm_pct,
    annot=False,
    fmt=".1f",
    cmap="Blues",
    linewidths=1.2,
    linecolor="white",
    xticklabels=labels,
    yticklabels=labels,
    vmin=0, vmax=100,
    cbar_kws={"label": "Row-normalised %", "shrink": 0.82},
    ax=ax
)

# annotate each cell: big count + small percentage
for i in range(2):
    for j in range(2):
        count = cm[i, j]
        pct   = cm_pct[i, j]
        txt_color = "white" if pct > 55 else "black"
        ax.text(j + 0.5, i + 0.38, f"{count}",
                ha="center", va="center",
                fontsize=22, fontweight="bold", color=txt_color)
        ax.text(j + 0.5, i + 0.65, f"({pct:.1f}%)",
                ha="center", va="center",
                fontsize=11, color=txt_color)

# diagonal label badges
diag_labels = ["True Negative", "True Positive"]
for k, (i, j, lbl) in enumerate([(0, 0, diag_labels[0]), (1, 1, diag_labels[1])]):
    ax.text(j + 0.5, i + 0.88, lbl,
            ha="center", va="center",
            fontsize=8.5, style="italic",
            color="white" if cm_pct[i,j] > 55 else GREY)

ax.set_xlabel("Predicted Label", fontsize=12, labelpad=8)
ax.set_ylabel("True Label",      fontsize=12, labelpad=8)
ax.set_title("Test-Set Confusion Matrix\n"
             r"(threshold $\tau$ = 0.168, frame-level, $n$ = 540)",
             fontsize=12, pad=12)

ax.tick_params(axis='both', length=0, labelsize=11)

# overall stats box
stats = (f"Accuracy : {(cm[0,0]+cm[1,1])/total*100:.2f}%\n"
         f"Macro-F1 : 0.604\n"
         f"AUC      : 0.637")
ax.text(1.32, 0.02, stats,
        transform=ax.transAxes,
        fontsize=8.5, va="bottom",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec=GREY, lw=0.8))

fig1.tight_layout()
fig1.savefig(r"C:\Users\DPQUAI250101\Desktop\deepfake_baseline_model\fig1_confusion_matrix.png",
             dpi=180, bbox_inches="tight")
print("✅  fig1_confusion_matrix.png saved")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIGURE 2 — Training / Validation Curves  (3 metrics × 2 stages)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── raw logged data ────────────────────────────────────────────────────────────
# Stage 1 (epochs 1-5)
s1_train_acc = [50.0, 54.3, 60.3, 60.0, 64.6]
s1_train_f1  = [0.343, 0.426, 0.538, 0.536, 0.603]
s1_train_auc = [0.654, 0.745, 0.797, 0.795, 0.837]
s1_val_acc   = [36.5, 33.8, 55.7, 43.0, 56.5]
s1_val_f1    = [0.312, 0.262, 0.557, 0.409, 0.565]
s1_val_auc   = [0.679, 0.684, 0.728, 0.730, 0.719]

# Stage 2 (epochs 1-5, early-stopped)
s2_train_acc = [71.9, 67.1, 69.8, 68.6, 69.6]
s2_train_f1  = [0.702, 0.650, 0.677, 0.662, 0.670]
# Stage 2 AUC not logged for train; use NaN
s2_train_auc = [np.nan]*5

s2_val_acc   = [52.5, 51.5, 51.2, 47.2, 49.8]
s2_val_f1    = [0.523, 0.509, 0.506, 0.459, 0.489]
s2_val_auc   = [0.744, 0.760, 0.769, 0.760, 0.774]

# ── build continuous x-axis: S1 = 1..5, S2 = 6..10 ───────────────────────────
x1 = np.arange(1, 6)
x2 = np.arange(6, 11)
x_all = np.concatenate([x1, x2])

train_acc = s1_train_acc + s2_train_acc
train_f1  = s1_train_f1  + s2_train_f1
train_auc = s1_train_auc + s2_train_auc
val_acc   = s1_val_acc   + s2_val_acc
val_f1    = s1_val_f1    + s2_val_f1
val_auc   = s1_val_auc   + s2_val_auc

# best checkpoint marker (S2 Ep1 → x=6)
best_ep_x = 6

# ── layout ────────────────────────────────────────────────────────────────────
fig2, axes = plt.subplots(1, 3, figsize=(13.5, 4.4), sharey=False)
fig2.subplots_adjust(wspace=0.34)

metric_cfg = [
    dict(
        ax=axes[0],
        title="Accuracy",
        ylabel="Accuracy (%)",
        train=train_acc,
        val=val_acc,
        best_marker=val_acc[best_ep_x - 1],
        ylim=(25, 80),
        yticks=range(30, 81, 10),
        scale=1.0,         # already in %
    ),
    dict(
        ax=axes[1],
        title="Macro-F1 Score",
        ylabel="Macro-F1",
        train=train_f1,
        val=val_f1,
        best_marker=val_f1[best_ep_x - 1],
        ylim=(0.20, 0.75),
        yticks=[0.2, 0.3, 0.4, 0.5, 0.6, 0.7],
        scale=1.0,
    ),
    dict(
        ax=axes[2],
        title="AUC-ROC",
        ylabel="AUC",
        train=train_auc,
        val=val_auc,
        best_marker=val_auc[best_ep_x - 1],
        ylim=(0.60, 0.90),
        yticks=[0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90],
        scale=1.0,
    ),
]

for cfg in metric_cfg:
    ax    = cfg["ax"]
    tr    = np.array(cfg["train"], dtype=float)
    vl    = np.array(cfg["val"],   dtype=float)

    # stage shading
    ax.axvspan(0.5, 5.5, alpha=0.07, color=BLUE,   zorder=0)
    ax.axvspan(5.5, 10.5, alpha=0.07, color=ORANGE, zorder=0)

    # stage divider
    ax.axvline(5.5, color=STAGE_DIV_COLOR, lw=1.2, ls="--", zorder=1)

    # curves
    ax.plot(x_all, tr, marker="o", ms=5, lw=1.8,
            color=BLUE,   label="Train", zorder=3)
    ax.plot(x_all, vl, marker="s", ms=5, lw=1.8,
            color=ORANGE, label="Val",   zorder=3)

    # mark best checkpoint
    bx  = best_ep_x
    bv  = cfg["best_marker"]
    ax.scatter([bx], [bv], s=120, zorder=5,
               color="none", edgecolors=RED, linewidths=2.0,
               label="Best ckpt")
    ax.annotate("best\nckpt", xy=(bx, bv),
                xytext=(bx + 0.55, bv + (cfg["ylim"][1]-cfg["ylim"][0])*0.07),
                fontsize=7.5, color=RED,
                arrowprops=dict(arrowstyle="->", color=RED, lw=1.0))

    # stage labels at top
    ax.text(3.0,  cfg["ylim"][1]*0.995, "Stage 1\n(head only)",
            ha="center", va="top", fontsize=8, color=BLUE, style="italic")
    ax.text(8.0,  cfg["ylim"][1]*0.995, "Stage 2\n(fine-tune)",
            ha="center", va="top", fontsize=8, color=ORANGE, style="italic")

    # early-stop annotation (last real point)
    ax.axvline(10, color=RED, lw=0.9, ls=":", alpha=0.7)
    ax.text(10.05, cfg["ylim"][0] + (cfg["ylim"][1]-cfg["ylim"][0])*0.04,
            "early\nstop", fontsize=7, color=RED, va="bottom")

    ax.set_title(cfg["title"], fontsize=12, pad=8)
    ax.set_xlabel("Epoch",     fontsize=10)
    ax.set_ylabel(cfg["ylabel"], fontsize=10)
    ax.set_xlim(0.5, 10.8)
    ax.set_ylim(*cfg["ylim"])
    ax.set_xticks(x_all)
    ax.set_xticklabels(
        [str(i) for i in range(1,6)] + [str(i) for i in range(1,6)],
        fontsize=8.5
    )
    ax.set_yticks(cfg["yticks"])
    ax.tick_params(axis='y', labelsize=8.5)
    ax.grid(axis='y', ls='--', alpha=0.45)
    ax.spines[["top","right"]].set_visible(False)

# shared legend below all panels
handles = [
    mpatches.Patch(color=BLUE,   label="Train"),
    mpatches.Patch(color=ORANGE, label="Validation"),
    plt.Line2D([0],[0], marker='o', color='none', ms=9,
               markeredgecolor=RED, markeredgewidth=2, label="Best checkpoint"),
]
fig2.legend(handles=handles, loc="lower center", ncol=3,
            fontsize=9.5, frameon=True, bbox_to_anchor=(0.5, -0.08))

fig2.suptitle(
    "Training & Validation Curves — MobileNetV2 Deepfake Detector (FF++ c23)",
    fontsize=12.5, y=1.02
)

fig2.savefig(r"C:\Users\DPQUAI250101\Desktop\deepfake_baseline_model/fig2_training_curves.png",
             dpi=180, bbox_inches="tight")
print("✅  fig2_training_curves.png saved")

✅  fig1_confusion_matrix.png saved
✅  fig2_training_curves.png saved
